
# CuPyCCx — CCD Computational Scaling Study

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varunrishi/CuPyCCx/blob/master/examples/colab_scaling.ipynb)

**Before running:** set runtime to GPU via *Runtime → Change runtime type → A100 GPU*

CCD (Coupled Cluster Doubles) has a formal computational cost of **O(N²_occ · N⁴_vir)**,
dominated by the T₂ · W_vvvv contraction. This notebook empirically measures that scaling
using a hydrogen chain series H₂ → H₄ → H₈ → H₁₆ → H₃₂ → H₆₄ → H₁₂₈ (atom count doubling
each step) in the STO-3G basis, where n_occ = n_vir = N (spin-orbitals) exactly.

| System | n_occ = n_vir | ERI tensor | Requires |
|--------|--------------|------------|----------|
| H2–H64 | 2–64 | 2 KB – 2 GB | any runtime |
| H128   | 128  | ~34 GB      | A100 80 GB  |

Two backends are compared:
- **C++ / cuBLAS**: compiled extension dispatching to cuBLAS DGEMMs (`cupyccx.method.CCD`)
- **PyCCD / CuPy**: pure-Python einsum solver on GPU (`cupyccx.py_solver.PyCCD`)

Timing uses 5 fixed iterations per system with convergence checks disabled,
so the measured time reflects pure residual-contraction cost.


In [ ]:
# Cell 1 — confirm GPU is available
!nvidia-smi
!nvcc --version

In [ ]:
%%bash
# Cell 2 — install dependencies; remove any stale cupyccx install
apt-get install -qq cmake ninja-build libeigen3-dev libopenblas-dev
pip install -q --upgrade pip
pip install -q pybind11 pyscf
pip uninstall -q -y cupyccx 2>/dev/null || true

In [ ]:

%%bash
# Cell 3 — clone and build with CUDA (A100 = sm_80; change to sm_75 for T4)
# Always start from /content so re-running doesn't nest CuPyCCx/CuPyCCx/...
cd /content

rm -rf CuPyCCx
git clone --quiet https://github.com/varunrishi/CuPyCCx.git
cd CuPyCCx

# Use the system cmake from apt (/usr/bin/cmake), NOT the pip-installed cmake
# which cannot locate nvcc. pip install pyscf pulls in cmake 3.31 as a
# Python package and it shadows the system cmake on PATH.
/usr/bin/cmake -B build \
  -DCUPYCCX_CUDA=ON \
  -DCUPYCCX_CUDA_ARCH=80 \
  -DCMAKE_BUILD_TYPE=Release \
  -DCUPYCCX_BUILD_TESTS=OFF \
  -Dpybind11_DIR=$(python3 -c "import pybind11; print(pybind11.get_cmake_dir())")
/usr/bin/cmake --build build -j$(nproc)

# Install Python files + CUDA extension directly into site-packages
SITE=$(python3 -c "import site; print(site.getsitepackages()[0])")
rm -rf "$SITE/cupyccx"
cp -r python/cupyccx "$SITE/"
cp build/_cupyccx*.so "$SITE/cupyccx/"
echo "Installed to: $SITE/cupyccx/"
ls "$SITE/cupyccx/"

# Verify import works before leaving bash
python3 -c "import importlib; importlib.invalidate_caches(); import cupyccx._cupyccx; print('Extension OK:', cupyccx._cupyccx.__file__)"


In [ ]:
# Cell 4 — verify extension loads in the notebook kernel
import sys, importlib, os, glob

# Evict all stale cupyccx modules
for key in list(sys.modules):
    if 'cupyccx' in key:
        del sys.modules[key]

# Find the installed .so and promote its site-packages to the front of sys.path.
# This handles the case where a stale editable install elsewhere on sys.path
# shadows the freshly built extension.
matches = glob.glob('/usr/local/lib/python*/dist-packages/cupyccx/_cupyccx*.so')
if matches:
    site_dir = os.path.dirname(os.path.dirname(matches[0]))
    sys.path = [site_dir] + [p for p in sys.path if p != site_dir]
    print(f'Using site-packages: {site_dir}')
else:
    print('WARNING: could not find _cupyccx*.so under /usr/local/lib — Cell 3 may not have completed')

importlib.invalidate_caches()

import cupyccx._cupyccx
print('Extension loaded from:', cupyccx._cupyccx.__file__)

In [ ]:
%%bash
# Cell 4b — install CuPy (matches Colab's CUDA 12.x runtime)
pip install -q cupy-cuda12x
python3 -c "import cupy; print('CuPy', cupy.__version__, '— CUDA', cupy.cuda.runtime.runtimeGetVersion())"

In [ ]:
# Cell 5 — imports, GPU detection, SCF data, warmup
import time, psutil
import numpy as np
from pyscf import gto, scf
from cupyccx.scf_data import prepare_from_pyscf
from cupyccx.method import CCD, CCOptions
from cupyccx.py_solver import PyCCD

# Detect GPU
try:
    import subprocess
    subprocess.run(['nvidia-smi'], check=True, capture_output=True)
    USE_GPU = True
except Exception:
    USE_GPU = False
print(f'USE_GPU = {USE_GPU}')

SCALING_ITERS = 5    # fixed iterations per system (no convergence)
WALL_LIMIT_S  = 300  # per-sweep wall-time guard (seconds)
N_ATOMS       = [2, 4, 8, 16, 32, 64, 128]  # doubles each step

def build_hchain(n_atoms):
    atom = '; '.join(f'H 0 0 {i * 1.4}' for i in range(n_atoms))
    mol  = gto.M(atom=atom, basis='sto-3g', unit='Bohr', verbose=0)
    mf   = scf.RHF(mol)
    mf.verbose = 0
    mf.kernel()
    assert mf.converged, f'HF did not converge for H{n_atoms}'
    return prepare_from_pyscf(mf, verbose=False)

# Pre-build SCFInputData; skip systems whose ERI exceeds 70% of available RAM.
# H128 needs ~34 GB — runs on A100 80 GB, skipped on T4/free Colab.
print('Building SCF data...')
scf_data = {}
for n in N_ATOMS:
    n_mo_so   = 2 * n                          # spin-orbital MOs for H_n
    eri_bytes = n_mo_so**4 * 8
    avail_ram = psutil.virtual_memory().available
    if eri_bytes > avail_ram * 0.7:
        print(f'  H{n:<3}  skipped: ERI {eri_bytes/1e9:.1f} GB > available RAM {avail_ram/1e9:.1f} GB')
        continue
    scf_data[n] = build_hchain(n)
    d = scf_data[n]
    print(f'  H{n:<3}  n_occ={d.n_occ:3d}  n_vir={d.n_vir:3d}  ERI={eri_bytes/1e9:.3f} GB')

built = sorted(scf_data.keys())

# --- Warmup: pre-JIT both backends before timed sweeps ---
opts_warm = CCOptions(use_gpu=USE_GPU, max_iter=3, conv_energy=0.0, conv_amp=0.0, use_diis=False)
d0 = scf_data[built[0]]

CCD.from_scf_data(d0, opts=opts_warm).compute(e_scf=d0.e_scf, verbose=False)
print('C++ warmup done.')

# PyCCD warmup: runs a short CuPy einsum to trigger JIT compilation of all
# kernel variants before timing starts — without this H2 shows ~3s of JIT overhead.
PyCCD.from_scf_data(d0, opts=opts_warm).compute(e_scf=d0.e_scf, verbose=False)
print('PyCCD warmup done.')

In [ ]:
# Cell 6 — C++ / cuBLAS backend scaling sweep
print('=== C++ / cuBLAS backend ===')
results_cpp = []
t_start = time.perf_counter()

for n_atoms in built:
    if time.perf_counter() - t_start > WALL_LIMIT_S:
        print(f'Wall limit reached, skipping H{n_atoms} and larger.')
        break

    data = scf_data[n_atoms]
    opts = CCOptions(use_gpu=USE_GPU, max_iter=SCALING_ITERS,
                     conv_energy=0.0, conv_amp=0.0, use_diis=False)

    t0 = time.perf_counter()
    CCD.from_scf_data(data, opts=opts).compute(e_scf=data.e_scf, verbose=False)
    dt = time.perf_counter() - t0

    print(f'  H{n_atoms:<3}  n_vir={data.n_vir:3d}  wall={dt:.3f}s')
    results_cpp.append({'label': f'H{n_atoms}', 'n_vir': data.n_vir,
                        'scale': data.n_occ**2 * data.n_vir**4, 'dt': dt})

print(f'Total: {time.perf_counter() - t_start:.1f}s')

In [ ]:
# Cell 7 — PyCCD / CuPy backend scaling sweep
print('=== PyCCD / CuPy backend ===')
results_py = []
t_start = time.perf_counter()

for n_atoms in built:
    if time.perf_counter() - t_start > WALL_LIMIT_S:
        print(f'Wall limit reached, skipping H{n_atoms} and larger.')
        break

    data = scf_data[n_atoms]
    opts = CCOptions(use_gpu=USE_GPU, max_iter=SCALING_ITERS,
                     conv_energy=0.0, conv_amp=0.0, use_diis=False)

    t0 = time.perf_counter()
    PyCCD.from_scf_data(data, opts=opts).compute(e_scf=data.e_scf, verbose=False)
    dt = time.perf_counter() - t0

    print(f'  H{n_atoms:<3}  n_vir={data.n_vir:3d}  wall={dt:.3f}s')
    results_py.append({'label': f'H{n_atoms}', 'n_vir': data.n_vir,
                       'scale': data.n_occ**2 * data.n_vir**4, 'dt': dt})

print(f'Total: {time.perf_counter() - t_start:.1f}s')

In [ ]:

# Cell 8 — CPU strong scaling: OMP_NUM_THREADS sweep on H32
# Measures how well the C++ / OpenBLAS backend parallelises on CPU cores.
# Uses use_gpu=False regardless of USE_GPU so the thread count is meaningful.
import ctypes, os

# OpenBLAS has already initialised its thread pool at import time, so
# os.environ changes alone are ignored. Call openblas_set_num_threads()
# via ctypes to change the count between runs.
try:
    _oblas = ctypes.cdll.LoadLibrary('libopenblas.so')
    def set_num_threads(n):
        _oblas.openblas_set_num_threads(n)
except OSError:
    _oblas = None
    def set_num_threads(n):
        os.environ['OMP_NUM_THREADS']      = str(n)
        os.environ['OPENBLAS_NUM_THREADS'] = str(n)

n_cpu = os.cpu_count()
print(f'Logical CPUs available: {n_cpu}')

# Powers of 2 up to n_cpu, always include n_cpu itself
thread_counts = sorted(set(
    [2**i for i in range(int(np.log2(max(n_cpu, 1))) + 1) if 2**i <= n_cpu] + [n_cpu]
))
print(f'Thread sweep: {thread_counts}')

data_h32   = scf_data[32]
opts_cpu   = CCOptions(use_gpu=False, max_iter=SCALING_ITERS,
                       conv_energy=0.0, conv_amp=0.0, use_diis=False)

# Warmup at full thread count
set_num_threads(n_cpu)
CCD.from_scf_data(data_h32, opts=opts_cpu).compute(e_scf=data_h32.e_scf, verbose=False)
print('CPU warmup done.')

results_threads = []
for n_threads in thread_counts:
    set_num_threads(n_threads)
    t0 = time.perf_counter()
    CCD.from_scf_data(data_h32, opts=opts_cpu).compute(e_scf=data_h32.e_scf, verbose=False)
    dt = time.perf_counter() - t0
    results_threads.append({'n_threads': n_threads, 'dt': dt})
    print(f'  threads={n_threads:2d}  wall={dt:.3f}s')

# Compute speedup relative to single-thread run
t1 = results_threads[0]['dt']
for r in results_threads:
    r['speedup'] = t1 / r['dt']
    print(f"  threads={r['n_threads']:2d}  speedup={r['speedup']:.2f}x")

# Restore full thread count for any subsequent cells
set_num_threads(n_cpu)

# --- Plot ---
import matplotlib.pyplot as plt

threads  = [r['n_threads'] for r in results_threads]
speedups = [r['speedup']   for r in results_threads]
times    = [r['dt']        for r in results_threads]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(threads, speedups, 'o-', color='steelblue', lw=2, label='measured')
ax.plot(threads, threads,  '--', color='gray', alpha=0.5, label='ideal (linear)')
ax.set_xlabel('OMP_NUM_THREADS', fontsize=11)
ax.set_ylabel('Speedup vs 1 thread', fontsize=11)
ax.set_title('H32 / CPU strong scaling  (C++ / OpenBLAS)', fontsize=11)
ax.legend(); ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot(threads, times, 's-', color='darkorange', lw=2)
ax2.set_xlabel('OMP_NUM_THREADS', fontsize=11)
ax2.set_ylabel(f'Wall time / {SCALING_ITERS} iters (s)', fontsize=11)
ax2.set_title('H32 wall time vs threads', fontsize=11)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('thread_scaling.png', dpi=150)
plt.show()


In [ ]:

# Cell 9 — CPU efficiency: how much of peak FLOP/s do the scalar loops extract?
#
# Strategy:
#   1. Compute theoretical FLOP count for one CCD iteration on H32.
#   2. Use the single-thread time from the OMP sweep above.
#   3. Benchmark numpy DGEMM (OpenBLAS) for the same matrix sizes — this is the
#      BLAS upper bound achievable without changing the C++ algorithm.
#   4. Report efficiency vs (a) theoretical CPU peak and (b) numpy BLAS.
import os
import numpy as np
import time as _time

# --- FLOP count per CCD iteration on H_N (o = v = N) ---
# Derived from ccd.cpp CPU path (build_W_oooo, build_W_vvvv, contract_* loops,
# Q_D two-pass intermediate, Fock/Q_C/Q_A sub-dominant terms):
#   build_W_oooo          : 2 * o^4 * v^2
#   build_W_vvvv          : 2 * o^2 * v^4
#   contract_klij_klab    : 2 * o^4 * v^2
#   contract_abcd_ijcd    : 2 * o^2 * v^4
#   contract_kbcj_ikac x4 : 8 * o^3 * v^3
#   Q_D (two-pass)        : 2*o^3*v^3 + 8*o^3*v^3  = 10 * o^3 * v^3
#   Fock/Q_C/Q_A          : O(N^5), sub-dominant — included approximately
def ccd_flops_per_iter(o, v):
    return (2*o**4*v**2 + 2*o**2*v**4   # build_W_oooo + build_W_vvvv
          + 2*o**4*v**2 + 2*o**2*v**4   # contract_klij + contract_abcd
          + 8*o**3*v**3                  # contract_kbcj x4
          + 10*o**3*v**3                 # Q_D two-pass
          + 4*o**2*v**3 + 4*o**3*v**2)  # Fock P(ab)/P(ij) ≈ O(N^5)

o32, v32 = 32, 32
flops_per_iter = ccd_flops_per_iter(o32, v32)
total_flops    = SCALING_ITERS * flops_per_iter
print(f'H32  o={o32}  v={v32}')
print(f'FLOPs per iteration : {flops_per_iter/1e9:.2f} GFLOPs')
print(f'FLOPs total (×{SCALING_ITERS} iters): {total_flops/1e9:.2f} GFLOPs')

# --- Achieved GFLOP/s from single-thread run ---
if not results_threads:
    print('Run the OMP sweep cell first.')
else:
    t_1thread = results_threads[0]['dt']   # wall time at OMP_NUM_THREADS=1
    gflops_achieved = total_flops / t_1thread / 1e9
    print(f'\nSingle-thread wall time : {t_1thread:.3f}s')
    print(f'Achieved GFLOP/s        : {gflops_achieved:.3f}')

    # --- Theoretical CPU peak (single core) ---
    # Colab CPUs are typically Intel Xeon Cascade Lake / Ice Lake
    # Estimate from /proc/cpuinfo if available, else assume 2.2 GHz
    try:
        with open('/proc/cpuinfo') as f:
            for line in f:
                if 'cpu MHz' in line:
                    ghz = float(line.split(':')[1].strip()) / 1000.0
                    break
    except Exception:
        ghz = 2.2
    # AVX-512 FMA: 2 FMAs/cycle × 8 FP64/register = 16 FLOPs/cycle/core
    # Scalar FMA: 2 FLOPs/cycle
    peak_avx512 = ghz * 16   # GFLOP/s per core with AVX-512
    peak_scalar = ghz * 2    # GFLOP/s per core scalar
    print(f'\nCPU clock              : {ghz:.2f} GHz')
    print(f'Theoretical peak (scalar, 1 core) : {peak_scalar:.1f} GFLOP/s')
    print(f'Theoretical peak (AVX-512, 1 core): {peak_avx512:.1f} GFLOP/s')
    print(f'Efficiency vs scalar peak : {100*gflops_achieved/peak_scalar:.1f}%')
    print(f'Efficiency vs AVX-512 peak: {100*gflops_achieved/peak_avx512:.1f}%')

    # --- NumPy DGEMM baseline for the same matrix size ---
    # contract_abcd_ijcd maps to: R[(o²,v²)] += W[(v²,v²)] @ T2[(v²,o²)]
    # For H32: (1024,1024) @ (1024,1024) — same FLOP count as the dominant loop.
    # numpy uses OpenBLAS with AVX-512; this is the best achievable without
    # changing the algorithm.
    oo, vv = o32**2, v32**2
    A = np.random.randn(vv, vv).astype(np.float64)
    B = np.random.randn(vv, oo).astype(np.float64)

    # Ensure single-thread for fair comparison
    set_num_threads(1)
    # Warmup
    _ = A @ B
    # Time
    N_REPS = 5
    t0 = _time.perf_counter()
    for _ in range(N_REPS):
        C = A @ B
    t_np = (_time.perf_counter() - t0) / N_REPS
    # Restore
    set_num_threads(n_cpu)

    dgemm_flops  = 2 * vv * vv * oo          # 2MNK FLOPs
    gflops_numpy = dgemm_flops / t_np / 1e9
    print(f'\nNumPy DGEMM ({vv}×{vv})×({vv}×{oo}), 1 thread:')
    print(f'  Wall time    : {t_np*1e3:.2f} ms')
    print(f'  GFLOP/s      : {gflops_numpy:.1f}')
    print(f'  Efficiency vs AVX-512 peak: {100*gflops_numpy/peak_avx512:.1f}%')
    print(f'\nC++ loops vs NumPy DGEMM: {100*gflops_achieved/gflops_numpy:.1f}%')
    print(f'  → switching dominant contractions to cblas_dgemm could give '
          f'~{gflops_numpy/gflops_achieved:.0f}× speedup on H32 CPU')


In [ ]:
# Cell 8 — results table
def fmt_row(r):
    return f'{r["label"]:<8} {r["n_vir"]:>5} {r["scale"]:>12.2e} {r["dt"]:>9.3f}'

hdr = f'{"System":<8} {"n_vir":>5} {"N^6_scale":>12} {"wall(s)":>9}'
sep = '-' * 40

print('C++ / cuBLAS')
print(hdr); print(sep)
for r in results_cpp:
    print(fmt_row(r))

print()
print('PyCCD / CuPy')
print(hdr); print(sep)
for r in results_py:
    print(fmt_row(r))

In [ ]:
# Cell 9 — log-log scaling plot: C++ vs PyCCD
import matplotlib.pyplot as plt

def loglog_fit(x, y):
    c = np.polyfit(np.log10(x), np.log10(y), 1)
    return c[0], c[1]  # slope, intercept

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax_idx, (xlabel, xkey) in enumerate([
    (r'$n_{occ}^2 \cdot n_{vir}^4$ (formal CCD cost)', 'scale'),
    (r'$n_{vir}$ (virtual spin-orbitals)',               'n_vir'),
]):
    ax = axes[ax_idx]
    ideal_slope = 1.0 if xkey == 'scale' else 6.0

    for results, color, marker, name in [
        (results_cpp, 'steelblue',  'o', 'C++ / cuBLAS'),
        (results_py,  'darkorange', 's', 'PyCCD / CuPy'),
    ]:
        if not results:
            continue
        xs = np.array([r[xkey] for r in results])
        ys = np.array([r['dt'] for r in results])
        slope, ic = loglog_fit(xs, ys)

        ax.scatter(xs, ys, color=color, marker=marker, s=70, zorder=5,
                   label=f'{name}  (slope={slope:.2f})')
        for r in results:
            ax.annotate(r['label'], (r[xkey], r['dt']),
                        textcoords='offset points', xytext=(5, 2), fontsize=8,
                        color=color)
        x_fit = np.logspace(np.log10(xs.min()), np.log10(xs.max()), 200)
        ax.plot(x_fit, 10**(slope * np.log10(x_fit) + ic), '--', color=color, alpha=0.6)

    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel(f'Wall time / {SCALING_ITERS} iters (s)', fontsize=11)
    ax.set_title(f'ideal slope = {ideal_slope:.0f}', fontsize=11)
    ax.legend(fontsize=9); ax.grid(True, which='both', alpha=0.3)

fig.suptitle(f'CCD scaling — {"GPU" if USE_GPU else "CPU"}  |  H-chain / STO-3G',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('scaling_plot.png', dpi=150)
plt.show()

## Interpreting the results

- **Ideal slope = 1.0** on the N²_occ · N⁴_vir axis (or **6.0** on the N_vir axis)
  confirms the dominant W_vvvv contraction drives the cost as O(N⁶).
- **Slope < ideal at small sizes** (H2–H8): GPU kernel-launch overhead and
  memory-bandwidth limits dominate when matrices are tiny.
- **Slope → ideal at large sizes** (H32–H128): once DGEMMs fill the GPU's SMs,
  the empirical exponent converges to 6.
- **C++ vs PyCCD gap**: the C++ backend dispatches hand-tuned cuBLAS DGEMMs;
  PyCCD uses CuPy `einsum` which incurs higher Python overhead and intermediate
  tensor allocations — the gap widens at large N where memory traffic matters most.